In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Compile Data

In [57]:
DATA_DIR = Path("../DATA")
csv_files = sorted(DATA_DIR.glob('*.csv'))

data_list = []
y_tensor = None

for i, file in enumerate(csv_files):
    df = pd.read_csv(file)

    # Extract 288 features from end and convert to numpy
    only_features = df.iloc[:,-288:]
    features_np = only_features.to_numpy()

    data_list.append(features_np)

    # labels will be same for every file, so just extract from the first one
    if i == 0:
        labels = df.iloc[:,0]

        # If 'NSEP', label is 0
        # Otherwise, label is 1 (gt10, gt30, gt60, gt100)
        y_tensor = np.where(labels == 'NSEP', 0, 1)

X_tensor = np.stack(data_list, axis=-1)

In [58]:
counts = np.bincount(y_tensor)
print(f"Unique labels: {df.iloc[:,0].unique().tolist()}")
print(f"Total NSEP: {counts[0]}")
print(f"Total SEP:  {counts[1]}")
print()
print(f"X_tensor shape: {X_tensor.shape}")  
print(f"y_tensor shape: {y_tensor.shape}")

Unique labels: ['NSEP', 'gt100', 'gt10', 'gt60', 'gt30']
Total NSEP: 17625
Total SEP:  169

X_tensor shape: (17794, 288, 10)
y_tensor shape: (17794,)


# Train/Val/Test Split

In [59]:
seed = 55

# Separate 60% for training. Other 40% for val/test
X_train, X_temp, y_train, y_temp = train_test_split(
    X_tensor, y_tensor,
    test_size=0.40,
    random_state=seed,
    stratify=y_tensor
)

#Separate val/test in half
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    random_state=seed,
    stratify=y_temp
)

print(f"Train set shape: {X_train.shape} | SEP count: {np.sum(y_train == 1)}")
print(f"Val set shape:   {X_val.shape}  | SEP count: {np.sum(y_val == 1)}")
print(f"Test set shape:  {X_test.shape}  | SEP count: {np.sum(y_test == 1)}")

Train set shape: (10676, 288, 10) | SEP count: 101
Val set shape:   (3559, 288, 10)  | SEP count: 34
Test set shape:  (3559, 288, 10)  | SEP count: 34


# Standard Normalization

In [61]:
# Reshape data to 2D for normalization
n_cases_train, n_time_train, n_features_train = X_train.shape
n_cases_val, n_time_val, n_features_val = X_val.shape 
n_cases_test, n_time_test, n_features_test = X_test.shape 

X_train_reshaped = X_train.reshape(-1, n_features_train )
X_val_reshaped = X_val.reshape(-1, n_features_val )
X_test_reshaped = X_test.reshape(-1, n_features_test )

# Apply standard scaling
scaler = StandardScaler()
X_train_reshaped = scaler.fit_transform(X_train_reshaped) 
X_val_reshaped = scaler.transform(X_val_reshaped) 
X_test_reshaped = scaler.transform(X_test_reshaped) 

# Reshape back to the original 3D shape
X_train = X_train_reshaped.reshape(n_cases_train, n_time_train, n_features_train)
X_val = X_val_reshaped.reshape(n_cases_val, n_time_val, n_features_val)
X_test = X_test_reshaped.reshape(n_cases_test, n_time_test, n_features_test)

print("Normalized X_train shape:", X_train.shape) 
print("Normalized X_val shape:  ", X_val.shape)   
print("Normalized X_test shape: ", X_test.shape)  
print()
print("X_Train Mean:", np.mean(X_train, axis=(0, 1)))
print("X_Train Std per feature: ", np.std(X_train, axis=(0, 1)))


Normalized X_train shape: (10676, 288, 10)
Normalized X_val shape:   (3559, 288, 10)
Normalized X_test shape:  (3559, 288, 10)

X_Train Mean: [ 1.06166267e-14 -2.52987579e-15  2.37958911e-14 -2.65694275e-14
  4.13989713e-15 -3.57936982e-16 -8.96203818e-16 -7.07104515e-16
 -1.32045900e-15  6.19217115e-17]
X_Train Std per feature:  [1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
